In [ ]:
!pip install transformers torch accelerate datasets

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    pad_token_id=tokenizer.eos_token_id
).to(device)

print("✅ GPT-2 loaded successfully!")

In [ ]:
import torch
print(torch.cuda.is_available())  # Should print: True

In [ ]:
PROMPT = "Artificial Intelligence is transforming the world by"

def generate(prompt, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(inputs["input_ids"], max_new_tokens=100, **kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print("✅ Helper function ready! Let's generate text!")


In [ ]:
print("METHOD 1 — Greedy Search")
print("-" * 60)
output = generate(PROMPT, do_sample=False)
print(output)

In [ ]:
print("METHOD 2 — Beam Search")
print("-" * 60)
output = generate(
    PROMPT,
    num_beams=5,
    no_repeat_ngram_size=2,
    early_stopping=True
)
print(output)

In [ ]:
print("METHOD 3 — Top-K Sampling")
print("-" * 60)
output = generate(PROMPT, do_sample=True, top_k=50, temperature=0.7)
print(output)

In [ ]:
print("METHOD 4 — Top-P / Nucleus Sampling")
print("-" * 60)
output = generate(PROMPT, do_sample=True, top_p=0.92, top_k=0, temperature=0.85)
print(output)

In [ ]:
print("METHOD 5 — Combined (Best Quality)")
print("-" * 60)
output = generate(
    PROMPT,
    num_beams=4, do_sample=True,
    top_k=50, top_p=0.90,
    temperature=0.75,
    no_repeat_ngram_size=3,
    early_stopping=True
)
print(output)

In [ ]:
print("PIPELINE — 3 unique generated sequences")
print("-" * 60)
gen_pipe = pipeline("text-generation", model="gpt2",
                    device=0 if device == "cuda" else -1)

results = gen_pipe(
    PROMPT,
    max_new_tokens=80,
    num_return_sequences=3,
    do_sample=True,
    top_k=50, top_p=0.92, temperature=0.8
)
for i, r in enumerate(results, 1):
    print(f"\n[Sequence {i}]\n{r['generated_text']}")

In [ ]:
from transformers import pipeline

print("PIPELINE — 3 unique generated sequences")
print("-" * 60)

gen_pipe = pipeline("text-generation", model="gpt2",
                    device=0 if device == "cuda" else -1)

results = gen_pipe(
    PROMPT,
    max_new_tokens=80,
    num_return_sequences=3,
    do_sample=True,
    top_k=50, top_p=0.92, temperature=0.8
)

for i, r in enumerate(results, 1):
    print(f"\n[Sequence {i}]\n{r['generated_text']}")

In [ ]:
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

PROMPT = "Artificial Intelligence is transforming the world by"

print("PIPELINE — 3 unique generated sequences")
print("-" * 60)

gen_pipe = pipeline("text-generation", model="gpt2",
                    device=0 if device == "cuda" else -1)

results = gen_pipe(
    PROMPT,
    max_new_tokens=80,
    num_return_sequences=3,
    do_sample=True,
    top_k=50, top_p=0.92, temperature=0.8
)

for i, r in enumerate(results, 1):
    print(f"\n[Sequence {i}]\n{r['generated_text']}")

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)

# Step 1 — Create custom corpus
corpus = """
Artificial intelligence is the simulation of human intelligence by machines.
Machine learning enables systems to learn and improve from experience.
Deep learning uses neural networks to model complex patterns in data.
Natural language processing allows computers to understand human language.
Generative AI creates new content such as text, images, and audio.
GPT-2 is a transformer-based language model developed by OpenAI.
Fine-tuning adapts a pre-trained model to a specific task or domain.
Transfer learning leverages knowledge from one domain to improve another.
The transformer architecture revolutionized NLP research in 2017.
Large language models are trained on massive corpora of internet text.
Attention mechanisms allow models to focus on relevant parts of the input.
Prompt engineering is the art of crafting inputs to guide model behavior.
"""

with open("corpus.txt", "w") as f:
    f.write(corpus.strip())
print("✅ Custom corpus saved!")
print(f"   Total words: {len(corpus.split())}")

# Step 2 — Dataset class
class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size=64):
        with open(file_path) as f:
            text = f.read()
        ids = tokenizer(text, return_tensors="pt")["input_ids"].squeeze()
        self.examples = [
            ids[i:i+block_size]
            for i in range(0, len(ids)-block_size, block_size)
        ]

    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        return {"input_ids": self.examples[i], "labels": self.examples[i]}

# Step 3 — Load model & tokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
ft_tokenizer = AutoTokenizer.from_pretrained("gpt2")
ft_model     = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
print("✅ Model loaded for fine-tuning!")

# Step 4 — Prepare dataset
dataset  = TextDataset("corpus.txt", ft_tokenizer)
collator = DataCollatorForLanguageModeling(tokenizer=ft_tokenizer, mlm=False)
print(f"✅ Dataset ready! Total blocks: {len(dataset)}")

# Step 5 — Training arguments
args = TrainingArguments(
    output_dir="./gpt2_ft",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=5,
    save_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available()
)

# Step 6 — Train!
trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=dataset,
    data_collator=collator
)

print("\n🚀 Starting fine-tuning...")
trainer.train()

# Step 7 — Save model
ft_model.save_pretrained("./gpt2_ft")
ft_tokenizer.save_pretrained("./gpt2_ft")
print("\n✅ Fine-tuning complete! Model saved to './gpt2_ft'")

In [ ]:
import os
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)

# Step 1 — Create custom corpus
corpus = """
Artificial intelligence is the simulation of human intelligence by machines.
Machine learning enables systems to learn and improve from experience.
Deep learning uses neural networks to model complex patterns in data.
Natural language processing allows computers to understand human language.
Generative AI creates new content such as text, images, and audio.
GPT-2 is a transformer-based language model developed by OpenAI.
Fine-tuning adapts a pre-trained model to a specific task or domain.
Transfer learning leverages knowledge from one domain to improve another.
The transformer architecture revolutionized NLP research in 2017.
Large language models are trained on massive corpora of internet text.
Attention mechanisms allow models to focus on relevant parts of the input.
Prompt engineering is the art of crafting inputs to guide model behavior.
"""

with open("corpus.txt", "w") as f:
    f.write(corpus.strip())
print("✅ Custom corpus saved!")

# Step 2 — Dataset class
class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size=64):
        with open(file_path) as f:
            text = f.read()
        ids = tokenizer(text, return_tensors="pt")["input_ids"].squeeze()
        self.examples = [
            ids[i:i+block_size]
            for i in range(0, len(ids)-block_size, block_size)
        ]
    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        return {"input_ids": self.examples[i], "labels": self.examples[i]}

# Step 3 — Load model & tokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
ft_tokenizer = AutoTokenizer.from_pretrained("gpt2")

# ✅ THIS IS THE FIX — set padding token
ft_tokenizer.pad_token = ft_tokenizer.eos_token

ft_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
ft_model.resize_token_embeddings(len(ft_tokenizer))
print("✅ Model loaded for fine-tuning!")

# Step 4 — Prepare dataset
dataset  = TextDataset("corpus.txt", ft_tokenizer)
collator = DataCollatorForLanguageModeling(
    tokenizer=ft_tokenizer,
    mlm=False
)
print(f"✅ Dataset ready! Total blocks: {len(dataset)}")

# Step 5 — Training arguments
args = TrainingArguments(
    output_dir="./gpt2_ft",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=5,
    save_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available()
)

# Step 6 — Train!
trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=dataset,
    data_collator=collator
)

print("\n🚀 Starting fine-tuning...")
trainer.train()

# Step 7 — Save model
ft_model.save_pretrained("./gpt2_ft")
ft_tokenizer.save_pretrained("./gpt2_ft")
print("\n✅ Fine-tuning complete! Model saved to './gpt2_ft'")

In [ ]:
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print(" FINE-TUNED MODEL — Text Generation")
print("=" * 60)

ft_pipe = pipeline(
    "text-generation",
    model="./gpt2_ft",
    tokenizer="./gpt2_ft",
    device=0 if device == "cuda" else -1
)

prompts = [
    "Deep learning is",
    "Natural language processing allows",
    "The transformer architecture"
]

for prompt in prompts:
    result = ft_pipe(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        top_p=0.92,
        temperature=0.8
    )[0]["generated_text"]
    print(f"\nPrompt : {prompt}")
    print(f"Output : {result}")
    print("-" * 60)

print("\n🎉 Task-01 Complete!")
print("   Text Generation with GPT-2 — Prodigy Infotech")